### 1. Overview of ML pipeline with PyTorch - Part 1 - Data

![alt text](<../imgs/Screenshot 2026-06-19 at 7.37.12 AM.png>)

A dataset is a collection of data. In practical settings, this can become very large and it is often not possible to load the entire dataset into memory. One of the solutions to this problem is to load the data in batches. A batch is a small subset of the dataset. But we need to preprocess and serve the data in order for our model to learn from it. This is where 3 important pytorch features come into play:
`Dataset`, `transform`, and `DataLoader`.

1. Dataset: Pytorch provides a `Dataset` class which is an abstract class representing a dataset. Your custom dataset should inherit `Dataset` and override the following methods:
- `__len__` so that `len(dataset)` returns the size of the dataset.
- `__getitem__` to support the indexing such that `dataset[i]` can be used to get the i-th sample.

2. Transform: Transforms are common image transformations provided by `torchvision`. They can be chained together using `transforms.Compose`. Some of the common transforms are:
- `transforms.ToTensor()`: Converts a PIL Image or numpy.ndarray to tensor.
- `transforms.Normalize(mean, std)`: Normalizes a tensor image with mean and standard deviation. The image is normalized as: `output[channel] = (input[channel] - mean[channel]) / std[channel]`
- transform.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)]): Composes several transforms together.

3. DataLoader: The `DataLoader` class provides an iterable over the given dataset. It can be used to load the data in batches, shuffle the data, and use multiprocessing for data loading. Some of the important parameters of `DataLoader` are:
- `dataset`: The dataset from which to load the data.
- `batch_size`: How many samples per batch to load.



#### Lecture Notes:
1. Transforms:Operations applied to each data point as it is loaded.Prepare data for the model.Common transforms: 
- ToTensor: Converts data to PyTorch tensors and scales values between 0 and 1.
- Normalize: Centers data around 0 and scales by standard deviation.Use Compose to chain multiple transforms in order.

2. Dataset:Wraps your data and fetches samples on demand (not all at once).Manages data location, loading specific samples, total sample count.Applies transforms to each sample during loading.Supports train/test split and downloading data if needed.Enables indexing to retrieve individual samples.

3. DataLoader:Serves data in batches for efficient training.Requests one batch at a time from the Dataset.Batch size controls how many samples per batch.Supports shuffling data to improve model learning.These components work together to efficiently handle large datasets without overwhelming memory.


![image-2.png](<../imgs/Screenshot 2026-07-02 at 12.02.14 AM.png>)

### 2. Overview of ML pipeline with PyTorch - Part 2 - Models

PyTorch Model Building and Training Pipeline

1. Model Creation with nn.Module:
    - Define layers in __init__ method (like gathering tools).
    - Define data flow in forward method (the path data takes).
    - Always call the model with input (e.g., model(input)), not model.forward(input).
    - Use super().__init__() in __init__ to enable PyTorch to track learnable parameters.
2. Training Loop Pattern:
    - zero_grad(): Clears old gradients.
    - backward(): Computes gradients.
    - step(): Updates model parameters.
    - Order matters; incorrect order can cause silent training failures.
3. Evaluation Mode:
    - Use model.eval() to set evaluation mode (not to evaluate directly).
    - Use torch.no_grad() to disable gradient tracking during evaluation.
    - Evaluate on unseen data to measure true performance (e.g., accuracy).
    - Switch back to training mode with model.train() if continuing training.
- Accuracy Calculation: Accuracy = (Number of correct predictions) / (Total predictions).


> note: switch back to training mode with model.train() if continuing training.

![image.png](<../imgs/Screenshot 2026-07-02 at 12.05.46 AM.png>)


![image.png](<../imgs/Screenshot 2026-07-02 at 12.10.04 AM.png>)

![image.png](<../imgs/Screenshot 2026-07-02 at 12.11.45 AM.png>)

### 3. Loss
`Measure`, `Diagnose`, and `Update`

loss = loss_function(outputs, target) -> measure (how well the model is performing i..e how wrong your predictions are.)
loss.backward() -> diagnose (compute gradients for each parameter based on the loss)
optimizer.step() -> update (update model parameters based on the gradients computed in the previous step)

Example Loss Functions:

1. Mean Squared Error (MSE) Loss: Measures the average squared difference between predicted and actual values. Commonly used for regression tasks.
```python
mse_loss = nn.MSELoss()
```

2. Cross-Entropy Loss: Measures the difference between two probability distributions. Commonly used for multi-class classification tasks.
```python
cross_entropy_loss = nn.CrossEntropyLoss()
```

3. Binary Cross-Entropy Loss: Measures the difference between two probability distributions for binary classification tasks.
```python
binary_cross_entropy_loss = nn.BCELoss()
```

4. Hinge Loss: Used for "maximum-margin" classification, primarily for support vector machines.
```python
hinge_loss = nn.MultiMarginLoss()
``` 

5. Kullback-Leibler Divergence Loss: Measures how one probability distribution diverges from a second, expected probability distribution.
```python
kl_div_loss = nn.KLDivLoss()
```


What each loss functions punishes:
- MSE Loss: Punishes larger errors more than smaller ones due to squaring the differences (e.g., an error of 2 is punished 4 times more than an error of 1).
- Cross-Entropy Loss: Punishes incorrect classifications based on the predicted probability distribution (e.g., punishes overconfident wrong predictions more than less confident ones)
- Binary Cross-Entropy Loss: Similar to Cross-Entropy Loss, but specifically for binary classification tasks. Punishes incorrect predictions based on the predicted probability of the positive class.
- Hinge Loss: Punishes predictions that are on the wrong side of the margin. The further away from the correct side, the higher the loss.
- Kullback-Leibler Divergence Loss: Punishes the divergence between the predicted and expected probability distributions. The more the predicted distribution diverges from the expected one, the higher the loss.

![image.png](<../imgs/Screenshot 2026-07-07 at 11.23.13 PM.png>)

### 4. Optimizers and Gradients

> when we do loss.backward(), we are just computing the gradients of the loss with respect to the model parameters. THIS is crucial to understand that we are NOT actually updating the model parameters yet. The actual update happens when we call optimizer.step(), which uses the computed gradients to adjust the model parameters in the direction that minimizes the loss.

### 5. Device Management

Every tensor (i.e. data) and model lives on a specific device (CPU or GPU). To perform computations on a specific device, you need to move your tensors and models to that device. If tensors and models are not on the same device, you will encounter error during computaiton.

`RuntimeError: Expected all tensors to be on the same device`

- How to control where you data lives:
    - `tensor.to(device)`: Moves the tensor to the specified device.
    - `model.to(device)`: Moves the model to the specified device.
    - `device = torch.device("cuda" if torch.cuda.is_available() else "cpu")`: Automatically selects GPU if available, otherwise CPU.

Pytorch by default uses CPU for all computations. If you have a GPU available, you can take advantage of it to speed up your computations. To do this, you need to move your tensors and models to the GPU using the `.to(device)` method.


- Helpfull commands:
```python
# Check if GPU is available
torch.cuda.is_available() # this returns True if GPU is available, otherwise False

# or you can use the commonly used pattern for this
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# here cuda stands for nvidia GPUs which had CUPDA toolkit installed. If you have AMD GPUs, you can use ROCm toolkit and use "rocm" instead of 
# "cuda" in the above command. If you have apple silicon, you can use "mps" instead of "cuda" in the above command. If you have no GPU, it will default to CPU.

# Get the name of the GPU
torch.cuda.get_device_name(0)

# Get the number of GPUs available
torch.cuda.device_count()

```


- Moving tensors and models:

# Model to device 
```python
model = model.to(device)
```

# training loop 
```python
for inputs, targets, in dataloader:
    # Move input and targets to the device
    inputs, targets = inputs.to(device), targets.to(device)
```

# check the device 
```python
# Check the device of a tensor
print(tensor.device)

# check the dvice of a model
print(next(model.parameters()).device) # models are not directly on devices, but their parameters are. So we can check the device of the first parameter of the model to know where the model is located.
```

# common mistakes with .to(device):
```python 

.to(deivce) # this looks like it moves x but it does not. It returns a new tensor on the device but does not change the original tensor. So you need to assign it back to x like this:
x = x.to(device) # this moves x to the device and assigns it back to x
```


# complete training look with device management:
```python 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MyModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_function = nn.CrossEntropyLoss()

for inputs, targets in dataloader:
    inputs, targets = inputs.to(device), targets.to(device)
    
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = loss_function(outputs, targets)
    loss.backward()
    optimizer.step()
```


> Pattern: Choose the device upfront, move the model to that device and move the data to that device. 

# This ensures that all computations are done on the same device and avoids any device mismatch errors.


- GPU memory management:
```python
# Clear GPU memory
torch.cuda.empty_cache() # this clears the GPU memory cache. It does not free up the memory occupied by tensors. It is useful when you want to free up GPU memory for other processes.
```

- GPU Memory Monitoring:
```python
# Check GPU memory usage
torch.cuda.memory_allocated() # returns the current GPU memory usage by tensors in bytes.
torch.cuda.memory_reserved() # returns the current GPU memory reserved by the caching allocator in bytes.
```

- GPU Memory error:
```python
# CUDA out of memory error: This error occurs when the GPU runs out of memory while trying to allocate memory for tensors. 

# If you get a CUDA out of memory error, you can try the following:
# 1. Reduce the batch size
# 2. Use a smaller model
# 3. Use mixed precision training (if supported by your model and hardware)
```

